# PAYMENTS CLEANING

In [ ]:
import pandas as pd
payments = pd.read_csv("../Data/sanlam_raw_payments.csv")

payments.head()

In [2]:
# Check to see if the transaction date has any NA values 
payments['transaction_date'].isna().sum()


np.int64(162)

In [3]:
# Step 1: Create a parser for the known formats
def parse_transaction_date(date_str):
    if pd.isna(date_str):
        return pd.NaT

    date_str = str(date_str).strip()

    try:
        # DD/MM/YYYY
        if '/' in date_str and len(date_str.split('/')[0]) == 2:
            return pd.to_datetime(date_str, format='%d/%m/%Y')

        # MM-DD-YYYY
        elif '-' in date_str and len(date_str.split('-')[0]) == 2:
            return pd.to_datetime(date_str, format='%m-%d-%Y')

        # YYYY-MM-DD
        elif len(date_str.split('-')[0]) == 4:
            return pd.to_datetime(date_str, format='%Y-%m-%d')

        # YYYY/MM/DD
        elif len(date_str.split('/')[0]) == 4:
            return pd.to_datetime(date_str, format='%Y/%m/%d')

        return pd.NaT

    except:
        return pd.NaT

In [10]:
# Step 2: Parse the dates
payments['parse_transaction_date'] = (
    payments['transaction_date']
    .apply(parse_transaction_date)
)

In [11]:
payments.head()

,source_row_id,transaction_ref,policy_id,client_id,transaction_date,amount,transaction_type,currency,payment_method,payment_status,source_system,parse_transaction_date
0,SPAY-0000000001,TX-S-0000000001,POL000003392,CL00007794,2022-11-15,"1,826.09",Commission,R,NaN,Reversed,LegacyLedger,2022-11-15
1,SPAY-0000000002,TX-S-0000000002,POL000000796,CL00008081,2022-10-23,"R -7,370.85",Refund,ZAR,EFT,success,PaymentsHub,2022-10-23
2,SPAY-0000000003,TX-S-0000000003,POL000003511,CL00007728,10/03/2025,"R 4,318.19",EFT,ZAR,Card,Paid,LegacyLedger,2025-03-10
3,SPAY-0000000004,TX-S-0000000004,POL000012172,CL00001221,2025/03/21,"R 4,114.14\r\n",Debit Order,R,EFT,Successful,PaymentsHub,2025-03-21
4,SPAY-0000000005,TX-S-0000000005,POL000014430,CL00008773,2025/04/23,"ZAR 21,151.57",EFT,ZAR,Debit Order,Reversed,BankFeed,2025-04-23


In [ ]:
# Step 3: Find malformed dates
bad_dates = payments[
    payments['transaction_date'].notna() &
    payments['parse_transaction_date'].isna()
]

In [18]:
# Step 4: Count malformed dates
bad_dates.shape[0]

454

In [13]:
#Step 5
payments['transaction_date_std'] = (
    payments['parse_transaction_date']
    .dt.strftime('%d-%m-%Y')
)

In [15]:
payments.columns.tolist()

['source_row_id',
 'transaction_ref',
 'policy_id',
 'client_id',
 'transaction_date',
 'amount',
 'transaction_type',
 'currency',
 'payment_method',
 'payment_status',
 'source_system',
 'parse_transaction_date',
 'transaction_date_std']

In [32]:
payments.head(30)

,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system
0,SPAY-0000000001,TX-S-0000000001,POL000003392,2022-11-15,CL00007794,"1,826.09",Commission,R,NaN,Reversed,LegacyLedger
1,SPAY-0000000002,TX-S-0000000002,POL000000796,2022-10-23,CL00008081,"R -7,370.85",Refund,ZAR,EFT,success,PaymentsHub
2,SPAY-0000000003,TX-S-0000000003,POL000003511,2025-03-10,CL00007728,"R 4,318.19",EFT,ZAR,Card,Paid,LegacyLedger
3,SPAY-0000000004,TX-S-0000000004,POL000012172,2025-03-21,CL00001221,"R 4,114.14\r\n",Debit Order,R,EFT,Successful,PaymentsHub
4,SPAY-0000000005,TX-S-0000000005,POL000014430,2025-04-23,CL00008773,"ZAR 21,151.57",EFT,ZAR,Debit Order,Reversed,BankFeed
5,SPAY-0000000006,TX-S-0000000006,POL000008647,2025-03-09,CL00000156,NaN,Premium,USD,NaN,Failed,BankFeed
6,SPAY-0000000007,TX-S-0000000007,POL000000436,2024-06-23,CL00003805,"(-24,653.54)",Reversal,R,Cash,Successful,PaymentsHub
7,SPAY-0000000008,TX-S-0000000008,POL000024244,2022-05-31,CL00000252,5285.68,Premium,ZAR,Cash,Paid,PaymentsHub
8,SPAY-0000000009,TX-S-0000000009,POL000012945,2021-03-31,CL00007328,"20,646.63",Premium,ZAR,Debit Order,Reversed,LegacyLedger
9,SPAY-0000000010,TX-S-0000000010,POL000010985,2025-10-10,CL00004931,"ZAR -21,763.22",Commission Reversal,R,Card,Successful,BankFeed


In [ ]:
#Rename the transaction_date_std to transaction_date_new
payments.rename(columns={'transaction_date_std': 'transaction_date_new'}, inplace=True)

In [23]:
# Remove the parse_transaction_date coloumn 
payments.drop(columns=['parse_transaction_date'], inplace=True)

In [25]:
# Remove the transaction_date coloumn (original)
payments.drop(columns=['transaction_date'], inplace=True)

In [27]:
cols = payments.columns.tolist()

# Remove the column from its current position
cols.remove('transaction_date_new')

# Insert it into the 4th position
cols.insert(3, 'transaction_date_new')

# Reorder the dataframe
payments = payments[cols]

In [31]:
payments.dtypes

source_row_id                      str
transaction_ref                    str
policy_id                          str
transaction_date_new    datetime64[us]
client_id                          str
amount                             str
transaction_type                   str
currency                           str
payment_method                     str
payment_status                     str
source_system                      str
dtype: object

In [30]:
# Convert string to datetime 
payments['transaction_date_new'] = pd.to_datetime(
    payments['transaction_date_new'],
    format='%d-%m-%Y',
    errors='coerce'
)

In [ ]:
## Check the malformed dates (QUARTINIE ALL 616 ROWS)
payments['transaction_date_new'].isna().sum()

np.int64(616)

In [ ]:
# Check the currency distinct values 
payments['currency'].unique().tolist()

['ZAR', 'USD']

In [39]:
payments['currency'] = payments['currency'].replace('R', 'ZAR')

In [ ]:
# Check the distinct values for payment method. 
payments['payment_method'].unique().tolist()

[nan, 'EFT', 'Card', 'Debit Order', 'Cash']

In [46]:
payments['payment_method'].isna().sum()

np.int64(12020)

In [47]:
# Set 1202payment_method0 payment method to "Unknown"
payments['payment_method'] = payments['payment_method'].fillna('Unknown')

In [48]:
payments['payment_method'].head(10)

0        Unknown
1            EFT
2           Card
3            EFT
4    Debit Order
5        Unknown
6           Cash
7           Cash
8    Debit Order
9           Card
Name: payment_method, dtype: str

In [51]:
payments['payment_method'].unique().tolist()

['Unknown', 'EFT', 'Card', 'Debit Order', 'Cash']

In [ ]:
#Remove trailling spaces and whitespaces. 
payments['payment_method'] = payments['payment_method'].str.strip()

In [57]:
payments.head(10)

,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system
0,SPAY-0000000001,TX-S-0000000001,POL000003392,2022-11-15,CL00007794,"1,826.09",Commission,ZAR,Unknown,Reversed,LegacyLedger
1,SPAY-0000000002,TX-S-0000000002,POL000000796,2022-10-23,CL00008081,"-7,370.85",Refund,ZAR,EFT,success,PaymentsHub
2,SPAY-0000000003,TX-S-0000000003,POL000003511,2025-03-10,CL00007728,"4,318.19",EFT,ZAR,Card,Paid,LegacyLedger
3,SPAY-0000000004,TX-S-0000000004,POL000012172,2025-03-21,CL00001221,"4,114.14",Debit Order,ZAR,EFT,Successful,PaymentsHub
4,SPAY-0000000005,TX-S-0000000005,POL000014430,2025-04-23,CL00008773,"21,151.57",EFT,ZAR,Debit Order,Reversed,BankFeed
5,SPAY-0000000006,TX-S-0000000006,POL000008647,2025-03-09,CL00000156,NaN,Premium,USD,Unknown,Failed,BankFeed
6,SPAY-0000000007,TX-S-0000000007,POL000000436,2024-06-23,CL00003805,"(-24,653.54)",Reversal,ZAR,Cash,Successful,PaymentsHub
7,SPAY-0000000008,TX-S-0000000008,POL000024244,2022-05-31,CL00000252,5285.68,Premium,ZAR,Cash,Paid,PaymentsHub
8,SPAY-0000000009,TX-S-0000000009,POL000012945,2021-03-31,CL00007328,"20,646.63",Premium,ZAR,Debit Order,Reversed,LegacyLedger
9,SPAY-0000000010,TX-S-0000000010,POL000010985,2025-10-10,CL00004931,"-21,763.22",Commission Reversal,ZAR,Card,Successful,BankFeed


In [ ]:
# Check if the coloumns contain R or ZAR 
payments[
    payments['amount'].astype(str).str.contains('R|ZAR', na=False)
]

In [56]:
#Replace the R and ZAR from amounts 
payments['amount'] = payments['amount'].str.replace('ZAR', '', regex=False)
payments['amount'] = payments['amount'].str.replace('R', '', regex=False)
payments['amount'] = payments['amount'].str.strip()

In [ ]:
#Check the distinct values in transcation type
payments['transaction_type'].unique().tolist()

['Commission',
 'Refund',
 'EFT',
 'Debit Order',
 'Premium',
 'Reversal',
 'Commission Reversal']

In [ ]:
#Display the value counts for each transaction type
payments['transaction_type'].value_counts(dropna=False)

transaction_type
Premium                18015
Commission             12024
Debit Order            11952
EFT                     7741
Reversal                4766
Commission Reversal     3103
Refund                  2399
Name: count, dtype: int64